In [ ]:
import os, json, math, random, gc, time, ast, copy
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights, resnet18

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

# BirdCLEF 2026 Training v8 - ResNet18 + EfficientNet-B0 (Nudge Improvements)
## Changes from v7 (nudges session):
- **Mixup augmentation** (α=0.2) — mixes pairs of mel spectrograms + labels; gentle regularisation that improves generalisation without hurting like SpecAugment
- **Gradient clipping** (max_norm=1.0) — prevents loss spikes, improves training stability
- **AUC-based model selection** — checkpoints the model with best validation AUC (competition metric) instead of val_loss
- **v8 model filenames** — `resnet18_v8_fold_*.pt` / `efficientnet_v8_fold_*.pt`

In [ ]:
# === DATA PATHS & SPECIES ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"
TAXONOMY_CSV = "/kaggle/input/competitions/birdclef-2026/taxonomy.csv"

# Load complete species list from taxonomy.csv (234 species)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species = taxonomy_df["primary_label"].astype(str).tolist()

species_set = set(species)

print(f"\u2705 Loaded {len(species)} species from taxonomy.csv")
print(f"First 10 species: {species[:10]}")
print(f"Last 10 species: {species[-10:]}")

# Create species index mapping
idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

# Load training data
df = pd.read_csv(TRAIN_META_CSV)
print(f"\n\u2705 Loaded training data: {len(df)} samples")
print(f"Unique species in training: {df['primary_label'].nunique()}")

# Save species.json for inference
with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print(f"\u2705 Saved species.json with {n_classes} species")

In [ ]:
# === CONFIG ===
CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    epochs=20,
    lr=1e-3,
    patience=7,
    num_workers=0,
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    # v8 nudges
    mixup_alpha=0.2,   # Mixup interpolation strength; 0.0 disables mixup
    clip_grad=1.0,     # Max gradient norm; prevents loss spikes
)

print(f"Config: {CFG}")

In [ ]:
# === HELPER FUNCTIONS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(len(species), dtype="float32")
    p = str(primary_id)
    if p in idx: y[idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[idx[sid]] = 1.0
    return y

def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr // 2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

def mixup_batch(x, y, alpha=0.2):
    """Apply Mixup to a batch of mel spectrograms and multi-hot labels.

    Draws a mixing coefficient lam ~ Beta(alpha, alpha) for each sample.
    Blends each sample with a randomly chosen partner from the same batch.
    Using per-sample lam (rather than a single batch-level scalar) gives
    slightly better diversity.
    """
    if alpha <= 0.0:
        return x, y
    lam = np.random.beta(alpha, alpha, size=x.size(0)).astype(np.float32)
    lam = torch.from_numpy(lam).to(x.device)
    # Reshape for broadcasting: (B,) -> (B,1,1,1) for x and (B,1) for y
    lam_x = lam.view(-1, 1, 1, 1)
    lam_y = lam.view(-1, 1)
    idx_perm = torch.randperm(x.size(0), device=x.device)
    x_mix = lam_x * x + (1.0 - lam_x) * x[idx_perm]
    y_mix = lam_y * y + (1.0 - lam_y) * y[idx_perm]
    return x_mix, y_mix

print("\u2705 Helper functions defined (mixup_batch added)")

In [ ]:
# === PRECOMPUTE MELS FROM TRAIN_AUDIO ===
print("Precomputing mels from train_audio\u2026")

MEL_OUT_DIR = "/kaggle/working/mels_v8"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

for idx_, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    mel_name = row["filename"].replace("/", "_") + ".npy"
    mel_path = Path(MEL_OUT_DIR) / mel_name

    # Skip if already computed
    if mel_path.exists():
        continue

    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except Exception:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])
    np.save(mel_path, mel)

print(f"\u2705 Mels saved from train_audio: {MEL_OUT_DIR}")

In [ ]:
# === EXTRACT SEGMENTS FROM TRAIN_SOUNDSCAPES (Missing/Underrepresented Species) ===
# Same targeted extraction as v7 — fills species gaps with soundscape segments.

print("\n" + "=" * 70)
print("SOUNDSCAPE EXTRACTION: Targeting missing/underrepresented species")
print("=" * 70)

SOUNDSCAPE_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
SOUNDSCAPE_ANNO_CSV = "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv"

pseudo_df = pd.DataFrame()

try:
    soundscape_labels = pd.read_csv(SOUNDSCAPE_ANNO_CSV)
    print(f"Loaded {len(soundscape_labels)} soundscape annotations")
    print(f"Columns: {soundscape_labels.columns.tolist()}")

    train_audio_species = set(df["primary_label"].astype(str).unique())
    soundscape_species_col = None
    for col in ["primary_label", "species", "bird_id"]:
        if col in soundscape_labels.columns:
            soundscape_species_col = col
            break
    if soundscape_species_col is None:
        soundscape_species_col = soundscape_labels.columns[1]
    print(f"Using soundscape species column: '{soundscape_species_col}'")

    soundscape_all_species = set(soundscape_labels[soundscape_species_col].astype(str).unique())

    species_counts_audio = df["primary_label"].value_counts().to_dict()
    target_species = {
        sp for sp in soundscape_all_species
        if species_counts_audio.get(sp, 0) < 5
    }
    print(f"Species in soundscapes: {len(soundscape_all_species)}")
    print(f"Target species (missing/underrepresented in train_audio): {len(target_species)}")

    pseudo_data = []
    for idx_, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels),
                          desc="Processing soundscapes"):
        primary = str(row.get(soundscape_species_col, "unknown"))

        if primary not in target_species:
            continue

        soundscape_id = None
        for id_col in ["soundscape_id", "filename", "file_id"]:
            if id_col in row.index:
                soundscape_id = str(row[id_col])
                break
        if soundscape_id is None:
            soundscape_id = str(row.iloc[0])

        audio_path = None
        for ext in [".ogg", ".wav", ".flac"]:
            candidate = Path(SOUNDSCAPE_DIR) / f"{soundscape_id}{ext}"
            if candidate.exists():
                audio_path = candidate
                break
        if audio_path is None:
            continue

        try:
            y, sr0 = sf.read(str(audio_path), always_2d=False)
        except Exception:
            continue

        if sr0 != CFG["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
        if y.ndim == 2:
            y = y.mean(axis=1)
        y = y.astype(np.float32)

        segment_samples = CFG["sr"] * CFG["seconds"]
        total_segments = max(1, len(y) // segment_samples)

        n_sample = min(5, total_segments)
        seg_indices = np.random.choice(total_segments, n_sample, replace=False)

        for seg_idx in seg_indices:
            start = seg_idx * segment_samples
            end = start + segment_samples
            if end > len(y):
                continue

            segment = y[start:end].copy()
            mel = logmel_from_wave(segment, CFG["sr"])

            mel_name = f"soundscape_{soundscape_id}_seg_{seg_idx}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)

            secondary = row.get("secondary_labels", "[]")
            pseudo_data.append({
                "filename": mel_name.replace(".npy", ""),
                "primary_label": primary,
                "secondary_labels": secondary,
            })

    if pseudo_data:
        pseudo_df = pd.DataFrame(pseudo_data)
        print(f"\u2705 Extracted {len(pseudo_df)} segments for {len(target_species)} target species")
    else:
        print("\u26a0\ufe0f  No soundscape segments extracted — proceeding with train_audio only")

except FileNotFoundError as e:
    print(f"\u26a0\ufe0f  Soundscape CSV not found: {e}")
    print("Continuing with train_audio only...")
except Exception as e:
    print(f"\u26a0\ufe0f  Soundscape processing failed: {e}")
    print("Continuing with train_audio only...")

In [ ]:
# === RESNET18 ARCHITECTURE (unchanged from v7) ===

class ResNet18Audio(nn.Module):
    """ResNet18-based classifier for mel spectrograms. Proven architecture (0.648 baseline)."""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

print("\u2705 ResNet18Audio architecture defined")

In [ ]:
# === EFFICIENTNET-B0 ARCHITECTURE (unchanged from v7) ===

class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("\u2705 EfficientNetB0Audio architecture defined (pretrained ImageNet backbone)")

In [ ]:
# === CLEAN DATASET (No SpecAugment — proven to hurt performance) ===
# Mixup is applied in the training loop (not here) so the dataset stays simple.

class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, cfg: dict, train: bool):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.cfg = cfg
        self.train = train
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        fname = str(r["filename"])
        if not fname.endswith(".npy"):
            mel_name = fname.replace("/", "_") + ".npy"
        else:
            mel_name = fname
        mel_path = self.mel_root / mel_name
        mel = np.load(mel_path).astype("float32")

        T = mel.shape[1]
        W = self.win_frames

        if T <= W:
            pad = np.zeros((mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([mel, pad], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else max(0, (T - W) // 2)
            mel = mel[:, start:start + W]

        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)
        x = torch.from_numpy(mel)[None, ...]
        y = torch.from_numpy(row_to_multihot(r["primary_label"], r.get("secondary_labels", "[]"))).float()
        return x.float(), y

print("\u2705 ClipDataset defined (no SpecAugment; mixup applied in training loop)")

In [ ]:
# === PREPARE TRAINING DATA ===
device = torch.device(CFG["device"])

mel_root = Path(MEL_OUT_DIR)
available_mels = set(f.stem for f in mel_root.glob("*.npy"))

print(f"Available mel files: {len(available_mels)}")

train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))
train_df = train_df[train_df["filename"].isin(available_mels)].reset_index(drop=True)
print(f"Training samples from train_audio: {len(train_df)}")

if len(pseudo_df) > 0:
    train_df = pd.concat([train_df, pseudo_df], ignore_index=True)
    print(f"Added {len(pseudo_df)} soundscape segments for missing/underrepresented species")
    print(f"Total training samples: {len(train_df)}")
else:
    print("No soundscape segments added")

species_counts = {sp: 0 for sp in species}
for _, row in train_df.iterrows():
    primary = str(row["primary_label"])
    if primary in species_counts:
        species_counts[primary] += 1
    for sp in parse_secondary(row.get("secondary_labels", "[]")):
        if sp in species_counts:
            species_counts[sp] += 1

print(f"\n\u2705 Training setup complete:")
print(f"  Device: {device}")
print(f"  Classes: {n_classes}")
print(f"  Samples: {len(train_df)}")
print(f"  Species with data: {sum(1 for c in species_counts.values() if c > 0)}")

In [ ]:
# === 5-FOLD CV TRAINING: RESNET18 (v8 nudges) ===
# Nudge 1: Mixup augmentation (alpha=0.2) — blends pairs of samples for better generalisation
# Nudge 2: Gradient clipping (max_norm=1.0) — prevents loss spikes
# Nudge 3: AUC-based model selection — checkpoints on best val AUC (competition metric)

print("\n" + "=" * 70)
print("TRAINING: ResNet18Audio (v8 nudges: mixup + grad-clip + AUC selection)")
print("=" * 70)

all_scores_resnet = []
all_val_preds_resnet = []
all_val_targets = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    print(f"\n\U0001f504 Fold {fold_idx + 1}/5")

    train_fold = train_df.iloc[train_idx]
    val_fold   = train_df.iloc[val_idx]

    train_ds = ClipDataset(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds   = ClipDataset(val_fold,   MEL_OUT_DIR, CFG, train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=CFG["num_workers"])
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"])

    model = ResNet18Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)

    pos_weight = torch.ones(n_classes).to(device)
    for i_sp, sp in enumerate(species):
        pos_weight[i_sp] = min(10.0, len(train_df) / (3.0 * max(1, species_counts[sp])))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-5)

    best_val_auc  = -1.0   # v8: select on AUC (not val_loss)
    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None
    best_fold_preds  = None
    best_fold_targets = None

    for epoch in range(CFG["epochs"]):
        # --- Training with Mixup (v8 nudge) ---
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            # Apply mixup only during training
            x_mix, y_mix = mixup_batch(x, y, alpha=CFG["mixup_alpha"])
            optimizer.zero_grad()
            loss = criterion(model(x_mix), y_mix)
            loss.backward()
            # Gradient clipping (v8 nudge)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG["clip_grad"])
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()

        # --- Validation (no mixup, use original labels) ---
        model.eval()
        val_loss = 0.0
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()
                fold_preds.append(torch.sigmoid(logits).cpu().numpy())
                fold_targets.append(y.cpu().numpy())
        val_loss /= len(val_loader)

        fp = np.vstack(fold_preds)
        ft = np.vstack(fold_targets)
        auc_scores = []
        for j in range(n_classes):
            if ft[:, j].sum() > 0 and (1 - ft[:, j]).sum() > 0:
                auc_scores.append(roc_auc_score(ft[:, j], fp[:, j]))
        val_auc = np.mean(auc_scores) if auc_scores else 0.0

        # v8: checkpoint on best val AUC (aligned with competition metric)
        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state  = copy.deepcopy(model.state_dict())
            best_fold_preds   = fp
            best_fold_targets = ft
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}")

        if patience_counter >= CFG["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), f"/kaggle/working/resnet18_v8_fold_{fold_idx}.pt")
    all_scores_resnet.append(best_val_auc)
    all_val_preds_resnet.append(best_fold_preds)
    all_val_targets.append(best_fold_targets)
    print(f"  \u2705 Best val_auc: {best_val_auc:.4f}  (val_loss at best: {best_val_loss:.4f})")

print(f"\n\u2705 ResNet18 Training Complete: mean_val_auc={np.mean(all_scores_resnet):.4f} \u00b1 {np.std(all_scores_resnet):.4f}")

In [ ]:
# === 5-FOLD CV TRAINING: EFFICIENTNET-B0 (v8 nudges) ===
# Same nudges as ResNet18: mixup + gradient clipping + AUC-based model selection

print("\n" + "=" * 70)
print("TRAINING: EfficientNetB0Audio (v8 nudges: mixup + grad-clip + AUC selection)")
print("=" * 70)

all_scores_efficientnet = []
all_val_preds_efficientnet = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    print(f"\n\U0001f504 Fold {fold_idx + 1}/5")

    train_fold = train_df.iloc[train_idx]
    val_fold   = train_df.iloc[val_idx]

    train_ds = ClipDataset(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds   = ClipDataset(val_fold,   MEL_OUT_DIR, CFG, train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=CFG["num_workers"])
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"])

    model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)

    pos_weight = torch.ones(n_classes).to(device)
    for i_sp, sp in enumerate(species):
        pos_weight[i_sp] = min(10.0, len(train_df) / (3.0 * max(1, species_counts[sp])))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-5)

    best_val_auc  = -1.0
    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state  = None
    best_fold_preds   = None

    for epoch in range(CFG["epochs"]):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            x_mix, y_mix = mixup_batch(x, y, alpha=CFG["mixup_alpha"])
            optimizer.zero_grad()
            loss = criterion(model(x_mix), y_mix)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG["clip_grad"])
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()
                fold_preds.append(torch.sigmoid(logits).cpu().numpy())
                fold_targets.append(y.cpu().numpy())
        val_loss /= len(val_loader)

        fp = np.vstack(fold_preds)
        ft = np.vstack(fold_targets)
        auc_scores = []
        for j in range(n_classes):
            if ft[:, j].sum() > 0 and (1 - ft[:, j]).sum() > 0:
                auc_scores.append(roc_auc_score(ft[:, j], fp[:, j]))
        val_auc = np.mean(auc_scores) if auc_scores else 0.0

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            best_fold_preds  = fp
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}")

        if patience_counter >= CFG["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), f"/kaggle/working/efficientnet_v8_fold_{fold_idx}.pt")
    all_scores_efficientnet.append(best_val_auc)
    all_val_preds_efficientnet.append(best_fold_preds)
    print(f"  \u2705 Best val_auc: {best_val_auc:.4f}  (val_loss at best: {best_val_loss:.4f})")

print(f"\n\u2705 EfficientNet Training Complete: mean_val_auc={np.mean(all_scores_efficientnet):.4f} \u00b1 {np.std(all_scores_efficientnet):.4f}")

In [ ]:
# === COMPUTE THRESHOLDS (Uniform 0.5) ===
print("\n" + "=" * 70)
print("THRESHOLD CONFIGURATION - Uniform 0.5 (proven approach, unchanged from v7)")
print("=" * 70)

all_resnet_preds = np.vstack(all_val_preds_resnet)
all_eff_preds    = np.vstack(all_val_preds_efficientnet)
all_targets      = np.vstack(all_val_targets)

ensemble_preds = (all_resnet_preds + all_eff_preds) / 2.0

auc_scores = []
for j in range(n_classes):
    y_true  = all_targets[:, j]
    y_score = ensemble_preds[:, j]
    if y_true.sum() > 0 and (1 - y_true).sum() > 0:
        auc_scores.append(roc_auc_score(y_true, y_score))
mean_auc = np.mean(auc_scores) if auc_scores else 0.0

print(f"2-Model Ensemble Validation AUC: {mean_auc:.4f} (across {len(auc_scores)} species)")

optimal_thresholds = {sp: 0.5 for sp in species}

with open("/kaggle/working/optimal_thresholds_v8.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print(f"\u2705 Saved uniform thresholds (0.5) for {len(optimal_thresholds)} species")
print(f"   -> /kaggle/working/optimal_thresholds_v8.json")

In [ ]:
# === TRAINING SUMMARY ===
print("\n" + "=" * 70)
print("TRAINING SUMMARY - v8 (ResNet18 + EfficientNet-B0, Nudge Improvements)")
print("=" * 70)

print(f"\n\U0001f4ca RESNET18 RESULTS (AUC-based selection):")
print(f"  Mean Val AUC: {np.mean(all_scores_resnet):.4f} \u00b1 {np.std(all_scores_resnet):.4f}")

print(f"\n\U0001f4ca EFFICIENTNET-B0 RESULTS (AUC-based selection):")
print(f"  Mean Val AUC: {np.mean(all_scores_efficientnet):.4f} \u00b1 {np.std(all_scores_efficientnet):.4f}")

print(f"\n\U0001f4ca 2-MODEL ENSEMBLE AUC: {mean_auc:.4f}")

print(f"\n\u2705 v8 NUDGES APPLIED:")
print(f"  \u2713 Mixup augmentation (alpha={CFG['mixup_alpha']}) — blends spectrogram pairs, improves generalisation")
print(f"  \u2713 Gradient clipping (max_norm={CFG['clip_grad']}) — prevents loss spikes")
print(f"  \u2713 AUC-based model selection — checkpoints aligned with competition metric")

print(f"\n\U0001f4be SAVED ARTIFACTS:")
print(f"  - ResNet18:    resnet18_v8_fold_0-4.pt")
print(f"  - EfficientNet: efficientnet_v8_fold_0-4.pt")
print(f"  - Thresholds:  optimal_thresholds_v8.json")
print(f"  - Species:     species.json")

print(f"\n\u2705 Training Complete! Ready for v8 inference.")